# Tema 8 · Modelos de Hugging Face en GPU
*Curso «Google Colab» · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** usar modelos ya entrenados para varias tareas de lenguaje y después **ajustar uno** a tus datos. Aquí la
GPU gratis hace la diferencia: lo que en CPU tarda horas, en GPU toma minutos.

**Parte A:** galería de tareas con `pipeline`. **Parte B:** ajuste fino de un modelo para clasificar sentimiento.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
# La prueba automática de MiAtlas.cl define MIATLAS_PRUEBA=1 para correr con pocos datos. Tú no necesitas tocarla.
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO, "| prueba rápida:", PRUEBA_RAPIDA)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU: el cuaderno funciona, pero más lento. En Colab: Entorno de ejecución → Cambiar tipo → GPU.")

## Token de Hugging Face (opcional)

Los modelos de este tema son públicos: **no necesitan token**. Con un token gratuito de tu cuenta de Hugging Face las
descargas tienen límites más altos. Si lo tienes, guárdalo en el panel 🔑 **Secretos** con el nombre `HF_TOKEN` (tema 4):
la biblioteca lo encuentra sola.

In [ ]:
%pip install -q transformers datasets accelerate

## Parte A · Galería de tareas

`pipeline` junta en una línea el tokenizador, el modelo y el pre y posprocesamiento. `device=0` usa la primera GPU.

In [ ]:
from transformers import pipeline
GPU = 0 if DISPOSITIVO == "cuda" else -1

sentimiento = pipeline("text-classification", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student", device=GPU)
sentimiento(["Me encantó el curso, aprendí muchísimo", "La clase de hoy fue confusa y aburrida"])

In [ ]:
entidades = pipeline("token-classification", model="Davlan/bert-base-multilingual-cased-ner-hrl",
                     aggregation_strategy="simple", device=GPU)
entidades("Gabriela Mistral nació en Vicuña y recibió el Nobel en Estocolmo.")

In [ ]:
traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en", device=GPU)
traductor("La GPU acelera el entrenamiento de las redes neuronales.")

### Generar texto con un modelo de lenguaje pequeño

**Qwen2.5-0.5B-Instruct** (Apache 2.0) es un modelo de lenguaje chico (unos **500 millones** de parámetros) que
entiende instrucciones. En GPU responde en segundos.

In [ ]:
generador = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=GPU)
mensajes = [{"role": "user", "content": "Resume en dos frases para qué sirve una GPU en inteligencia artificial."}]
respuesta = generador(mensajes, max_new_tokens=120, do_sample=False)
print(respuesta[0]["generated_text"][-1]["content"])

> **Experimenta:** pídele un resumen de un párrafo tuyo, o que clasifique un texto como positivo o negativo. Un modelo
> de lenguaje hace muchas tareas «sin entrenar» (*zero-shot*), aunque peor que un modelo ajustado: lo verás en el
> proyecto final.

## Parte B · Ajuste fino (*fine-tuning*)

Tomamos **BETO**, un BERT entrenado con español por la Universidad de Chile (`dccuchile/bert-base-spanish-wwm-cased`),
y le enseñamos a clasificar tweets en **negativo, neutro y positivo**.

Los datos: la parte en español de *tweet_sentiment_multilingual* (CC BY 3.0, de Cardiff NLP), con 1.839 tweets para
entrenar. Son tweets reales: algunos traen groserías.

In [ ]:
from datasets import load_dataset

datos = load_dataset("mteb/tweet_sentiment_multilingual", "spanish")
ETIQUETAS = ["negativo", "neutro", "positivo"]
print(datos)
print(datos["train"][0])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE = "dccuchile/bert-base-spanish-wwm-cased"   # BETO
tokenizador = AutoTokenizer.from_pretrained(BASE)
modelo = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=3).to(DISPOSITIVO)

def preparar(lote):
    return tokenizador(lote["text"], truncation=True, max_length=96, padding="max_length")

tokens = datos.map(preparar, batched=True)
tokens = tokens.map(lambda f: {"labels": int(f["label"])})
tokens.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
from torch.utils.data import DataLoader
import time

def evaluar(modelo, particion):
    modelo.eval()
    correctas = 0
    with torch.no_grad():
        for lote in DataLoader(tokens[particion], batch_size=64):
            lote = {k: v.to(DISPOSITIVO) for k, v in lote.items()}
            correctas += (modelo(**lote).logits.argmax(-1) == lote["labels"]).sum().item()
    return correctas / len(tokens[particion])

print(f"Antes de entrenar (al azar): {evaluar(modelo, 'test'):.1%}")
optimizador = torch.optim.AdamW(modelo.parameters(), lr=3e-5)
for epoca in range(1, 4):
    modelo.train()
    inicio = time.perf_counter()
    for lote in DataLoader(tokens["train"], batch_size=32, shuffle=True):
        lote = {k: v.to(DISPOSITIVO) for k, v in lote.items()}
        perdida = modelo(**lote).loss
        perdida.backward()
        optimizador.step()
        optimizador.zero_grad()
    print(f"Época {epoca}: exactitud en validación {evaluar(modelo, 'validation'):.1%} · {time.perf_counter() - inicio:.0f} s")
print(f"Exactitud final en prueba: {evaluar(modelo, 'test'):.1%}")

## Pruébalo tú

Escribe `clasificar(texto)`, que use el modelo ajustado y devuelva `"negativo"`, `"neutro"` o `"positivo"`.
Pista: tokeniza el texto, pásalo por el modelo y toma el `argmax` de los `logits`.

In [ ]:
def clasificar(texto):
    # ← tu código aquí
    return None

In [ ]:
# Comprobación
r = clasificar("Qué día tan maravilloso, todo me salió perfecto")
if r is None:
    print("Aún no lo resuelves.")
elif r in ETIQUETAS:
    print("✅ Funciona. El modelo dice:", r)
else:
    print("❌ Debe devolver una de", ETIQUETAS)

## Resumen

- `pipeline(tarea, model=..., device=0)` usa un modelo listo en la GPU.
- Un **ajuste fino** corto adapta un modelo general a tu tarea, y en GPU toma minutos.
- Los modelos y datos de Hugging Face tienen **licencia**: revísala antes de usarlos.

**Fuentes:** [Transformers · pipelines](https://huggingface.co/docs/transformers/main_classes/pipelines) · [Transformers · ajuste fino](https://huggingface.co/docs/transformers/training) · [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) · [sentimiento multilingüe](https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student) · [entidades](https://huggingface.co/Davlan/bert-base-multilingual-cased-ner-hrl) · [traducción es→en](https://huggingface.co/Helsinki-NLP/opus-mt-es-en) · [BETO](https://github.com/dccuchile/beto) · [tweet_sentiment_multilingual](https://huggingface.co/datasets/mteb/tweet_sentiment_multilingual)